# Debate Topology: A Physics Question

This notebook demonstrates the **debate** multi-agent topology on a single physics question. Two of the three debaters use a small, inexpensive model and one uses a stronger model: can the strong debater convince the others, where a simple majority vote of their first answers would be wrong?

**How it works:**
- **Round 1**: several *debaters* answer the question independently
- **Later rounds**: each debater reads the other debaters' latest answers, re-examines its own, and gives an updated answer
- **Early stop**: the debate ends as soon as all debaters agree (or after `max_rounds` rounds)
- **Final answer**: a majority vote over the debaters' last answers

With `max_rounds=1` the agents never see each other, so the result is a plain **majority vote** over independent answers (*self-consistency*). Comparing the round-1 vote with the final answer therefore shows what the debate added.

## 1. Setup and Imports

In [1]:
import sys
import os
import json
from pathlib import Path
from IPython.display import Markdown, display

# If the notebook lives in examples/, make sure src/ is importable
sys.path.insert(0, str(Path.cwd().parent / "src"))

from mas_topologies import TopologyAgent, PromptSet
from mas_topologies.protocol import majority_vote, normalize_answer

## 2. API Configuration

Export your API key before starting the notebook:

```bash
export OPENAI_API_KEY="sk-..."
export ANTHROPIC_API_KEY="sk-ant-..."
export GEMINI_API_KEY="..."
```

The debaters below all use OpenAI models, so `OPENAI_API_KEY` is enough; the other keys are only needed if you change the models.

In [2]:
# Set your API keys here ONLY if they are not already exported in your environment.
# (Assigning placeholder strings here would override real keys and make every call fail;
#  also avoid saving real keys in a notebook that is committed to git.)
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["GEMINI_API_KEY"] = "..."

In [3]:
# Two debaters with a small model and one with a stronger model (only an OpenAI key is
# needed). The strong debater goes last, so that a tie-break (earliest debater wins)
# never favours it. Any provider works, e.g. ("anthropic", "claude-sonnet-4-5") or
# ("gemini", "gemini-2.5-flash"); debaters whose API key is not set are skipped.
DEBATER_MODELS = [
    ("openai", "gpt-4o-mini"),
    ("openai", "gpt-4o-mini"),
    ("openai", "gpt-5-mini"),
]
KEY_VARS = {"openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY", "gemini": "GEMINI_API_KEY"}

# Generic capabilities for models AutoGen's built-in list may not know
GENERIC_MODEL_INFO = {"vision": False, "function_calling": True, "json_output": True,
                      "family": "unknown", "structured_output": True,
                      "multiple_system_messages": False}

def build_client(provider, model):
    from autogen_ext.models.openai import OpenAIChatCompletionClient
    kwargs = {"model": model, "api_key": os.environ[KEY_VARS[provider]]}
    if provider != "openai":
        # Claude and Gemini through their OpenAI-compatible endpoints (works with the base install)
        kwargs.update(include_name_in_message=False, model_info=GENERIC_MODEL_INFO)
    return OpenAIChatCompletionClient(**kwargs)

available = [(p, m) for p, m in DEBATER_MODELS if os.environ.get(KEY_VARS[p])]
if not available:
    raise EnvironmentError("No API key found. Export at least one of: " + ", ".join(KEY_VARS.values()))

DEBATER_CLIENTS = [build_client(p, m) for p, m in available]
DEBATER_LABELS = [f"{p}/{m}" for p, m in available]
print("Debater models:", ", ".join(DEBATER_LABELS))

Debater models: openai/gpt-4o-mini, openai/gpt-4o-mini, openai/gpt-5-mini


## 3. Define the Prompt

The system message sets good working practice for the debaters: derive from first principles, check that approximations are valid, verify the result, and stay concise (long derivations make every later round expensive, because each debater reads the others' answers). The answer format asks for just the number, so that answers can be compared. The debate prompt itself (shown below) is the library's default.

In [4]:
PROMPT_SET = PromptSet(
    system_message=(
        "You are an expert physicist. Solve the problem from first principles: state your "
        "assumptions, check that every approximation you use is valid for the given values, "
        "and verify the result with a limiting case or a quick numerical estimate. "
        "Keep your response concise (at most about 250 words)."
    ),
    final_answer_hint=(
        "End your response with a line of the form 'FINAL ANSWER: <number>', giving only the "
        "number to 3 significant figures, with no units or comments."
    ),
)

print("Debate prompt sent to each debater from round 2 on:\n")
print(PROMPT_SET.debate_prompt)

Debate prompt sent to each debater from round 2 on:

Here are the most recent answers of the other agents (debate round {round}):

{others}Compare them with your own previous answer. Where the answers differ, identify the specific step or assumption that causes the difference and decide which version is correct. Change your answer only if you find a concrete error in your own reasoning; do not simply adopt the majority view. Be concise: give the decisive argument rather than repeating the whole derivation, then your COMPLETE updated answer.


## 4. The Question

A pendulum released at 90° is far outside the small-angle regime, so the familiar period $T_0 = 2\pi\sqrt{L/g}$ no longer applies. The exact period involves a complete elliptic integral: $T/T_0 = 2K(\sin 45°)/\pi \approx 1.180$. Answering "1" (the small-angle result) is a tempting mistake.

We ask for 3 significant figures, and compare the debaters' answers at that precision with a small custom `answer_normalizer` (so that, e.g., `1.85` and `1.854` count as the same answer).

In [5]:
QUESTION = (
    "A simple pendulum is released from rest at an angle of 90 degrees from the vertical. "
    "By what factor is its period longer than the small-angle period 2*pi*sqrt(L/g)? "
    "Give the factor to 3 significant figures."
)
EXPECTED = 1.18034   # 2 K(1/sqrt(2)) / pi

def three_sig_figs(answer):
    """Compare numeric answers at 3 significant figures (non-numbers: library default)."""
    text = normalize_answer(answer)
    try:
        return f"{float(text):.3g}"
    except ValueError:
        return text

def is_correct(answer, rel_tol=0.01):
    """True if the answer is a number within 1% of the expected factor."""
    try:
        return abs(float(normalize_answer(answer)) - EXPECTED) / EXPECTED < rel_tol
    except ValueError:
        return False

print(QUESTION)

A simple pendulum is released from rest at an angle of 90 degrees from the vertical. By what factor is its period longer than the small-angle period 2*pi*sqrt(L/g)? Give the factor to 3 significant figures.


## 5. Create and Run the Debate

- `n_experts=3`: three debaters
- `max_rounds=3`: at most three rounds (the debate stops earlier if all debaters agree)
- `expert_model_clients`: one model per debater (debater *i* uses client *i*)
- `answer_normalizer`: how answers are compared for consensus and voting
- the final answer is the majority vote (`aggregation="vote"`, the default)

In [6]:
LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

agent = TopologyAgent(
    model_client=DEBATER_CLIENTS[0],
    topology="debate",
    prompts=PROMPT_SET,
    n_experts=3,                           # number of debaters
    max_rounds=3,                          # maximum number of rounds
    expert_model_clients=DEBATER_CLIENTS,  # one model per debater
    answer_normalizer=three_sig_figs,      # compare answers at 3 significant figures
    log_dir=str(LOG_DIR),
)

result = await agent.run(QUESTION, metadata={"task": "pendulum_90deg", "topology": "debate",
                                             "models": DEBATER_LABELS})

print(f"\n{'='*80}")
print(f"Rounds used    : {result.rounds_used}")
print(f"Final answer   : {result.final_answer}")
print(f"{'='*80}")

[TopologyAgent] Session log: logs/session_20261102_005508.jsonl

[TopologyAgent] topology=debate
Running a debate: 3 debaters, up to 3 round(s), aggregation=vote, execute_code=False

=== Debate round 1 ===

--- Debater_1 (round 1) ---
Assumptions:
1. The pendulum swings without any damping or external forces.
2. The motion can be approximated as simple harmonic motion for small angles.

To find the factor by which the period of the pendulum at 90 degrees differs from the small-angle approximation, we can use the formula for the period of a simple pendulum, which in general is:

\[ T = 2\pi \sqrt{\frac{L}{g}} \]

For large angles, the period is given by:

\[ T = 2\pi \sqrt{\frac{L}{g}} \cdot K(k) \]

where \( K(k) \) is the complete elliptic integral of the first kind, and \( k = \sin(\theta/2) \). For \( \theta = 90^\circ \), \( k = 1 \) and \( K(1) = \frac{\pi}{2} \).

For \( \theta = 90^\circ \):

\[ T = 2\pi \sqrt{\frac{L}{g}} \cdot \frac{\pi}{2} = \pi^2 \sqrt{\frac{L}{g}} \]

The f

## 6. How the Answers Evolved

Each debater's final answer in every round (as compared for voting):

In [7]:
answers_by_round = result.info["answers_by_round"]
n = len(answers_by_round[0])

rows = ["| Round | " + " | ".join(f"Debater {i + 1} ({DEBATER_LABELS[i % len(DEBATER_LABELS)]})"
                                  for i in range(n)) + " |",
        "|---|" + "---|" * n]
for r, answers in enumerate(answers_by_round, 1):
    cells = [f"{a} {'✓' if is_correct(a) else '✗'}" for a in answers]
    rows.append(f"| {r} | " + " | ".join(cells) + " |")

display(Markdown("\n".join(rows)))

| Round | Debater 1 (openai/gpt-4o-mini) | Debater 2 (openai/gpt-4o-mini) | Debater 3 (openai/gpt-5-mini) |
|---|---|---|---|
| 1 | 1.57 ✗ | 2 ✗ | 1.18 ✓ |
| 2 | 2 ✗ | 2 ✗ | 1.18 ✓ |
| 3 | 1.18 ✓ | 2 ✗ | 1.18 ✓ |

## 7. Outcome: Vote and Consensus

In [8]:
info = result.info
print(f"Votes            : {info['votes']}")
print(f"Winner           : {info['winner']}" + ("  (no majority: tie broken by debater order)" if info["tie"] else ""))
print(f"Consensus reached: " + (f"in round {info['consensus_round']}" if info["consensus_round"] else "no"))
print(f"Final answer     : {result.final_answer}   -> {'correct' if is_correct(result.final_answer) else 'incorrect'}"
      f" (expected ≈ {EXPECTED:.3g})")

Votes            : {'1.18': 2, '2': 1}
Winner           : Debater_1
Consensus reached: no
Final answer     : 1.18   -> correct (expected ≈ 1.18)


## 8. Did the Debate Help?

The round-1 answers are independent, so their majority vote is what plain majority voting (self-consistency) would have returned. Comparing it with the final answer, and the cost of each, shows what the extra rounds of debate added on this question. If all round-1 answers differ, there is no majority to compare with.

In [9]:
# Majority vote over the round-1 answers = majority voting without any debate
r1_winner, r1_votes, r1_tie = majority_vote(answers_by_round[0], normalizer=lambda a: a)
r1_answer = None if r1_tie else answers_by_round[0][r1_winner]   # a tie is no majority

# Token cost: round 1 alone vs the whole debate (each message records its round and tokens)
def tokens(msgs):
    return sum(m.get("prompt_tokens", 0) + m.get("completion_tokens", 0) for m in msgs)

round1_tokens = tokens([m for m in result.messages if m.get("round") == 1])
total_tokens = result.usage["total"]["prompt_tokens"] + result.usage["total"]["completion_tokens"]

final_ok = is_correct(result.final_answer) and not info["tie"]
if r1_answer is None:
    print(f"Majority vote after round 1 : none (votes {r1_votes})")
else:
    print(f"Majority vote after round 1 : {r1_answer}  {r1_votes}  -> {'correct' if is_correct(r1_answer) else 'incorrect'}")
print(f"Final answer after debate   : {result.final_answer}" + ("  (no majority)" if info["tie"] else "")
      + f"  -> {'correct' if final_ok else 'incorrect'}")
print(f"Tokens: round 1 only = {round1_tokens},  whole debate = {total_tokens}  "
      f"({result.usage['total']['calls']} model calls)")

before = None if r1_answer is None else is_correct(r1_answer)
verdict = {
    (False, True): "The debate FIXED a wrong majority.",
    (True, False): "The debate turned a correct majority into a WRONG answer.",
    (True, True): "Both correct: on this question the debate only confirmed the vote.",
    (False, False): "Both wrong: the debate did not recover the correct answer.",
    (None, True): "Round 1 had NO majority; after the debate the majority answer was CORRECT.",
    (None, False): "Round 1 had NO majority; the debate did not reach the correct answer.",
}[(before, final_ok)]
print("\n" + verdict)

Majority vote after round 1 : none (votes {'1.57': 1, '2': 1, '1.18': 1})
Final answer after debate   : 1.18  -> correct
Tokens: round 1 only = 4131,  whole debate = 19013  (9 model calls)

Round 1 had NO majority; after the debate the majority answer was CORRECT.


## 9. Read the Debate

In [10]:
for m in result.messages:
    content = m["content"]
    print(f"\n--- {m['source']} (round {m['round']}) ---")
    print(content[:800] + ("..." if len(content) > 800 else ""))


--- Debater_1 (round 1) ---
Assumptions:
1. The pendulum swings without any damping or external forces.
2. The motion can be approximated as simple harmonic motion for small angles.

To find the factor by which the period of the pendulum at 90 degrees differs from the small-angle approximation, we can use the formula for the period of a simple pendulum, which in general is:

\[ T = 2\pi \sqrt{\frac{L}{g}} \]

For large angles, the period is given by:

\[ T = 2\pi \sqrt{\frac{L}{g}} \cdot K(k) \]

where \( K(k) \) is the complete elliptic integral of the first kind, and \( k = \sin(\theta/2) \). For \( \theta = 90^\circ \), \( k = 1 \) and \( K(1) = \frac{\pi}{2} \).

For \( \theta = 90^\circ \):

\[ T = 2\pi \sqrt{\frac{L}{g}} \cdot \frac{\pi}{2} = \pi^2 \sqrt{\frac{L}{g}} \]

The factor by which the period at \( 90^...

--- Debater_2 (round 1) ---
**Assumptions:**
1. The pendulum behaves as a simple pendulum.
2. The angle of release (90 degrees) is large enough to deviate significant

## 10. Inspect the Session Log

In [11]:
log_files = sorted(LOG_DIR.glob("session_*.jsonl"))
if log_files:
    latest_log = log_files[-1]
    print(f"Reading session log: {latest_log}")
    with open(latest_log, "r") as f:
        session = json.loads(f.readlines()[-1])   # JSONL: one line per run; take the latest

    print(f"  Topology     : {session['topology']}")
    print(f"  Rounds used  : {session['rounds_used']}")
    print(f"  Final answer : {session['final_answer']}  (tagged: {session['final_answer_tagged']})")
    print(f"  Info         : {json.dumps(session['info'])}")
    print(f"  Token usage  : {session['usage']['total']}")
else:
    print("No log files found")

Reading session log: logs/session_20261102_005508.jsonl
  Topology     : debate
  Rounds used  : 3
  Final answer : 1.18  (tagged: True)
  Info         : {"aggregation": "vote", "n_agents": 3, "consensus_round": null, "answers_by_round": [["1.57", "2", "1.18"], ["2", "2", "1.18"], ["1.18", "2", "1.18"]], "votes": {"1.18": 2, "2": 1}, "winner": "Debater_1", "tie": false}
  Token usage  : {'prompt_tokens': 11235, 'completion_tokens': 7778, 'calls': 9}


## Summary

This notebook demonstrated the **debate** topology:

**Key takeaways:**
- Debaters answer independently, then revise after reading each other's answers
- The debate stops early once all debaters agree; the final answer is a majority vote
- The round-1 vote equals plain majority voting, so comparing it with the final answer shows what the debate added, and the token counts show what it cost
- All rounds, votes and token counts are logged for reproducibility

**Things to try:**
- More debaters (`n_experts`) or rounds (`max_rounds`)
- Other models in `DEBATER_MODELS`, or a single model for all debaters
- `max_rounds=1` for plain majority voting, or `stop_on_consensus=False` to always debate every round
- `aggregation="judge"` for free-form answers, where a judge synthesises the final answer instead of a vote
- `execute_code=True` for problems that are best solved numerically: each debater's code is run, and the outputs are shared in the next round